In [ ]:
# importation des bibliothèques nécessaires
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException
import time
import pandas as pd
import os
import dateparser

# Configurer Selenium WebDriver pour automatiser la navigation
driver_path = 'C:/Users/traor/Desktop/Extraction_Linkedin/chromedriver.exe'


# Vérification de l'existence du fichier 'chromedriver.exe
if not os.path.exists(driver_path):
    raise FileNotFoundError(f"Le fichier chromedriver n'existe pas : {driver_path}")


# Création du service et initialisation du driver Selenium
service = Service(driver_path)
driver = webdriver.Chrome(service=service)

# URL de connexion LinkedIn
linkedin_login_url = 'https://www.linkedin.com/login'
username = '_ _ _ _ _'  # Remplacez par votre nom d'utilisateur LinkedIn
password = '_ _ _ _ _ '  # Remplacez par votre mot de passe LinkedIn

# Mots-clés de recherche
keywords = [
    "Protection des données Afrique",
    "Cyberattaques Afrique" 
    "Cybersécurité et IA en Afrique",
    "Sécurité informatique Afrique",
    "Intelligence artificielle Afrique",
    "IA en Afrique",
    "Technologies de sécurité Afrique",
    "IA cybersécurité Afrique",
    "Cybercriminalité Afrique",
    "Cyber défense Afrique",
    "Cyber menaces Afrique",
    "Sécurité des systèmes Afrique",
    "Conférence IA Afrique"
]

# Liste pour stocker les données extraites
posts_data = []

# Fonction pour convertir une date relative en format standard (datetime)
def convert_relative_date(relative_date):
    if relative_date:
        date = dateparser.parse(relative_date, settings={'RELATIVE_BASE': pd.Timestamp.now()})
        if date:
            return date.strftime('%Y-%m-%d %H:%M:%S')
    return None

# Fonction pour se connecter à LinkedIn via Selenium
def linkedin_login(driver, username, password):
    driver.get(linkedin_login_url)
    
    # Attente que les éléments de connexion soient visibles
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, 'username')))
    
    # Envoi des identifiants de connexion
    driver.find_element(By.ID, 'username').send_keys(username)
    driver.find_element(By.ID, 'password').send_keys(password)
    driver.find_element(By.XPATH, '//button[@type="submit"]').click()
    # Attente de la connexion
    time.sleep(3)
    
# Fonction pour trouver un élément en toute sécurité
def safe_find_element(parent, by, value):
    try:
        return parent.find_element(by, value)
    except NoSuchElementException:
        return None

# Fonction de scraping d'une page LinkedIn pour un mot-clé
def scrape_linkedin_page(driver, keyword):
    # Construction de l'URL de recherche LinkedIn avec le mot-clé
    url = f"https://www.linkedin.com/search/results/content/?keywords={keyword}&origin=GLOBAL_SEARCH_HEADER"
    driver.get(url)
    time.sleep(5)
    
    # défilé dans la page pour charger plus de résultats
    body = driver.find_element(By.TAG_NAME, "body")
    for _ in range(15):
        body.send_keys(Keys.PAGE_DOWN)
        time.sleep(1.5)
        
    # Récupération de tous les posts de la page
    posts = driver.find_elements(By.CLASS_NAME, "artdeco-card")
    print(f">>> {len(posts)} posts trouvés pour : {keyword}")
    
    # Parcours des posts et extraction des informations
    for post in posts:
        try:
            # Extraction du titre (contenu)
            title_elem = safe_find_element(post, By.CLASS_NAME, "update-components-actor__title")
            content = title_elem.text.strip() if title_elem else 'N/A'
            
            # Extraction des métadonnées : auteur et date
            meta_elem = safe_find_element(post, By.CLASS_NAME, "update-components-actor__meta")
            author_elem = safe_find_element(meta_elem, By.CLASS_NAME, "update-components-actor__title") if meta_elem else None
            author = author_elem.text.strip() if author_elem else 'N/A'
            
            # Extraction de la date de publication
            date_elem = safe_find_element(meta_elem, By.CLASS_NAME, "update-components-actor__sub-description")
            date_text = None
            if date_elem:
                spans = date_elem.find_elements(By.TAG_NAME, "span")
                for span in spans:
                    if any(x in span.text.lower() for x in ["il y a", "ago", "h", "j", "sem", "mois"]):
                        date_text = span.text.strip()
                        break
            date = convert_relative_date(date_text) if date_text else 'N/A'
            
            # Extraction des informations d'interaction : likes, commentaires, reposts
            like_elem = safe_find_element(post, By.CSS_SELECTOR, ".social-details-social-counts__reactions-count")
            commentaire_elem = safe_find_element(post, By.CSS_SELECTOR, ".social-details-social-counts__comments")
            repost_elem = safe_find_element(post, By.CSS_SELECTOR, "li.social-details-social-counts__comments + li")

            like = like_elem.text.strip() if like_elem else 'N/A'
            commentaire = commentaire_elem.text.strip() if commentaire_elem else 'N/A'
            repost = repost_elem.text.strip() if repost_elem else 'N/A'
            
            # Ajout des données extraites à la liste
            posts_data.append({
                'Keyword': keyword,
                'Author': author,
                'Date': date,
                'Like': like,
                'Commentaire': commentaire,
                'Repost': repost,
                'Title': content
            })

            print(f"[OK] {author} | {date} | Like: {like} | Repost: {repost} | Com: {commentaire}")

        except Exception as e:
            print(f"[ERREUR POST] {e}")

# Connexion à LinkedIn
linkedin_login(driver, username, password)

# Sauvegarde partielle des données extraites à chaque itération
for keyword in keywords:
    scrape_linkedin_page(driver, keyword)
    df_partial = pd.DataFrame(posts_data)
    df_partial.to_csv('Pub_linkedin_partial.csv', index=False)

# Sauvegarde finale des données extraites dans un fichier CSV
df_final = pd.DataFrame(posts_data)
df_final.to_csv('Pub_linkedin_final.csv', index=False)

# Fermeture du driver Selenium
driver.quit()

# Message de confirmation
print("Données extraites et enregistrées avec succès dans Pub_linkedin_final.csv")


>>> 21 posts trouvés pour : Protection des données Afrique
[OK] TSplus Group
TSplus Group | 2025-03-22 22:35:14 | Like: 7 | Repost: N/A | Com: N/A
[OK] Forum International sur la protection des données à caractère personnel (FIPDCP)
Forum International sur la protection des données à caractère personnel (FIPDCP) | None | Like: 60 | Repost: 8 republications | Com: 2 commentaires
[OK] Cyber Africa Forum (CAF)
Cyber Africa Forum (CAF) | None | Like: 158 | Repost: 7 republications | Com: 2 commentaires
[ERREUR POST] 'NoneType' object has no attribute 'find_element'
[OK] E-karangé
E-karangé | None | Like: 41 | Repost: N/A | Com: N/A
[OK] Ariel Dossa
Ariel Dossa
 • 3e et +
 • 3e et + | None | Like: 2 | Repost: 3 republications | Com: 2 commentaires
[OK] Jules Hervé Yimeumi
Jules Hervé Yimeumi
 • 3e et +
 • 3e et + | 2024-09-05 22:35:30 | Like: 46 | Repost: 7 republications | Com: 6 commentaires
[OK] Africa Data Protection (ADP)
Africa Data Protection (ADP) | 2024-09-05 22:35:30 | Like: N/A |

In [ ]:
# Nettoyage des données extraites
import pandas as pd

In [3]:
# Charger les données extraites depuis le fichier CSV
df_posts = pd.read_csv('Pub_linkedin_final.csv')

In [ ]:
# Inspection initiale des données
print("Aperçu des données:")
print(df_posts.head())

# Nettoyage des données
# Supprimer les doublons
df_posts.drop_duplicates(inplace=True)

# Gérer les valeurs manquantes
df_posts.fillna('N/A', inplace=True)

# Inspection après nettoyage
print("Données après nettoyage:")
print(df_posts.head())

# Stockage des données nettoyées dans un nouveau fichier CSV
df_posts.to_csv('Pub_linkedin_Pro_1.csv', index=False)

print("Données nettoyées et enregistrées dans Pub_linkedin_Pro_1.csv")

Aperçu des données:
                          Keyword  \
0  Protection des données Afrique   
1  Protection des données Afrique   
2  Protection des données Afrique   
3  Protection des données Afrique   
4  Protection des données Afrique   

                                              Author                 Date  \
0                         TSplus Group\nTSplus Group  2025-03-22 22:35:14   
1  Forum International sur la protection des donn...                  N/A   
2  Cyber Africa Forum (CAF)\nCyber Africa Forum (...                  N/A   
3                               E-karangé\nE-karangé                  N/A   
4   Ariel Dossa\nAriel Dossa\n • 3e et +\n • 3e et +                  N/A   

    Like     Commentaire            Repost  \
0    7.0             N/A               N/A   
1   60.0  2 commentaires  8 republications   
2  158.0  2 commentaires  7 republications   
3   41.0             N/A               N/A   
4    2.0  2 commentaires  3 republications   

                 